In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# Chọn 1 bảng bất kỳ để làm Test Data
test_name = "Emission_Nox"

try:
    df_raw_test = pd.read_csv(f"{test_name}.csv")
    print(f" Đọc thành công file Test: {test_name}.csv | Kích thước gốc: {df_raw_test.shape}")
    display(df_raw_test.head(3))
except FileNotFoundError:
    print(f" Không tìm thấy file {test_name}.csv. Vui lòng kiểm tra lại tên file.")

 Đọc thành công file Test: Emission_Nox.csv | Kích thước gốc: (17, 116)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Sox/ Khí Sox,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling \n(Month/Year),Stack \nTemperature,%Oxygen,Gas Velocity,Flow rate,SOx Emission\nActual O2,SOx Emission \nrate,...,Flow rate,SOx Emission\nActual O2,SOx Emission \nrate,Operation \nHour,SOx Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN


## Columns Description

---

### 1. Thông tin chung
* **`name`**: Tên sản phẩm. Bao gồm tên tiếng Anh và tiếng Việt (Ví dụ: Coal/ Than 5A).
* **`unit`**: Đơn vị tính. Ở đây là `Ton` (Tấn).
* **`name_no_vn`**: Tên sản phẩm đã lược bỏ tiếng Việt, chỉ giữ lại tiếng Anh hoặc mã hiệu để thuận tiện cho việc xử lý dữ liệu (Data processing).

### 2. Dữ liệu thời gian (Tháng)
Các cột từ `jan` đến `dec` đại diện cho khối lượng tiêu thụ/nhập kho tương ứng với 12 tháng trong năm:
| Cột | Tháng |
| :--- | :--- |
| **jan** | Tháng 1 |
| **feb** | Tháng 2 |
| **mar** | Tháng 3 |
| **apr** | Tháng 4 |
| **may** | Tháng 5 |
| **jun** | Tháng 6 |
| **jul** | Tháng 7 |
| **aug** | Tháng 8 |
| **sep** | Tháng 9 |
| **oct** | Tháng 10 |
| **nov** | Tháng 11 |
| **dec** | Tháng 12 |

### 3. Các ký hiệu đặc biệt trong dữ liệu
* **Số thập phân**: Sử dụng dấu chấm `.` để phân tách (Ví dụ: `1233.522` tấn).
* **`0.000`**: Giá trị bằng không (không có phát sinh trong tháng).
* **`NaN`**: (Not a Number) Dữ liệu trống hoặc chưa có thông tin ghi nhận.
* **Ký hiệu (DV), (YB)**: Đại diện cho các **Cơ sở phát thải (Emission Facilities)** hoặc chi nhánh. ESG yêu cầu báo cáo chi tiết theo từng địa điểm vận hành.

##BƯỚC 1 - Chuẩn hóa tên và loại bỏ tiếng Việt

In [ ]:
import pandas as pd
import re

# Từ điển ánh xạ cụm từ chỉ định sang tiếng Anh
translation_map = {
    "Sấy phun 1": "Spray Dryer 1",
    "Sấy phun 2": "Spray Dryer 2",
    "Ống khói lò nung xương 1,2": "Biscuit Kiln Exhaust Stack 1, 2",
    "Ống khói lò nung xương 3,4": "Biscuit Kiln Exhaust Stack 3, 4",
    "Ống khói lò nung xương 5,6": "Biscuit Kiln Exhaust Stack 5, 6",
    "Ống khói lò nung men số 1": "Glaze Kiln Exhaust Stack 1",
    "Ống khói lò nung men số 2": "Glaze Kiln Exhaust Stack 2",
    "Ống khói lò nung men số 3": "Glaze Kiln Exhaust Stack 3",
    "Ống khói lò nung men số 4": "Glaze Kiln Exhaust Stack 4",
    "Ống khói lò nung men số 5": "Glaze Kiln Exhaust Stack 5",
    "Ống khói lò nung men số 6": "Glaze Kiln Exhaust Stack 6",
    "Sampling \n(Month/Year)": "Sampling"
}

# Định nghĩa các ký tự tiếng Việt dùng cho Regex
viet_chars = (
    r'àáâãèéêìíòóôõùúýăđơư'
    r'ạảấầẩẫậắằẳẵặẹẻẽếềểễệ'
    r'ỉịọỏốồổỗộớờởỡợụủứừửữự'
    r'ỳỵỷỹÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ'
    r'ẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼẾỀỂỄỆ'
    r'ỈỊỌỎỐỒỔỖỘỚỜỞỠỢỤỦỨỪỬỮỰ'
    r'ỲỴỶỸ'
)

def process_and_remove_vietnamese(text):
    if not isinstance(text, str) or pd.isna(text):
        return text

    # Thay thế các cụm từ chỉ định trước
    for vi, en in translation_map.items():
        if vi in text:
            text = text.replace(vi, en)

    # Xóa phần tiếng Việt
    text = re.sub(rf'\s*\([^)]*[{viet_chars}][^)]*\)', '', text)
    text = re.sub(rf'\s*/[^/]*[{viet_chars}].*', '', text)
    text = re.sub(rf'\s+[{viet_chars}a-zA-Z]*[{viet_chars}][\w\s]*$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_vietnamese_in_table(df):
    new_df = df.copy()
    # Chỉ áp dụng hàm xử lý lên các cột có kiểu dữ liệu là chuỗi (object/string)
    string_cols = new_df.select_dtypes(include=['object', 'string']).columns

    for col in string_cols:
        new_df[col] = new_df[col].apply(process_and_remove_vietnamese)

    return new_df

# --- CHẠY TEST BƯỚC 1 ---
df_step1 = clean_vietnamese_in_table(df_raw_test)
print(f"--- KẾT QUẢ BƯỚC 1: Kích thước: {df_step1.shape} ---")
display(df_step1.head(10))

--- KẾT QUẢ BƯỚC 1: Kích thước: (17, 116) ---


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Sox,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling,Stack Temperature,%Oxygen,Gas Velocity,Flow rate,SOx Emission Actual O2,SOx Emission rate,...,Flow rate,SOx Emission Actual O2,SOx Emission rate,Operation Hour,SOx Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Spray Dryer 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,15.72,1.3362,...,1416.666667,94.2,8.007,660,5284.62,NaN,NaN,NaN,NaN,NaN
4,Spray Dryer 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,18.34,1.5589,...,1416.666667,94.7,8.0495,436,3509.582,NaN,NaN,NaN,NaN,NaN
5,"Biscuit Kiln Exhaust Stack 1, 2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.3333333,83.84,0.871936,...,154,92.4,0.853776,744,635.209344,NaN,NaN,NaN,NaN,NaN
6,"Biscuit Kiln Exhaust Stack 3, 4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.3333333,84.71,0.957223,...,158.3333333,96.1,0.91295,744,679.2348,NaN,NaN,NaN,NaN,NaN
7,"Biscuit Kiln Exhaust Stack 5, 6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,80.19,0.8444007,...,194.1666667,93.8,1.09277,744,813.02088,NaN,NaN,NaN,NaN,NaN
8,Glaze Kiln Exhaust Stack 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.6666667,6.38,0.0645656,...,178.3333333,96.2,1.02934,744,765.82896,NaN,NaN,NaN,NaN,NaN
9,Glaze Kiln Exhaust Stack 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,6.42,0.0545058,...,155.6666667,97.8,0.913452,744,679.608288,NaN,NaN,NaN,NaN,NaN


# BƯỚC 2 - Xóa các hàng và cột không cần thiết

In [ ]:
def clean_data_custom(df):
    """
    1. Xóa hàng có cột đầu tiên là NaN.
    2. Xóa các hàng mà cột đầu tiên bắt đầu bằng 'Total' hoặc 'Specific'.
    3. Xóa các cột có chứa chữ 'Process'.
    """
    first_col = df.columns[0]

    # NaN ở cột đầu tiên
    df = df.dropna(subset=[first_col])

    # Xóa hàng bắt đầu bằng "Total" hoặc "Specific"
    mask_to_drop = df[first_col].astype(str).str.startswith(('Total', 'Specific'))
    df = df[~mask_to_drop]

    # Tìm và xóa cột có chữ "Process"
    cols_to_drop = [col for col in df.columns if "Process" in str(col)]
    for col in df.columns:
        if col not in cols_to_drop:
            if df[col].astype(str).str.contains("Process", na=False).any():
                cols_to_drop.append(col)

    df_cleaned = df.drop(columns=cols_to_drop)

    return df_cleaned

# --- CHẠY TEST BƯỚC 2 ---
df_step2 = clean_data_custom(df_step1)
print(f"--- KẾT QUẢ BƯỚC 2: Kích thước: {df_step2.shape} ---")
display(df_step2.head(10))

--- KẾT QUẢ BƯỚC 2: Kích thước: (13, 115) ---


,Unnamed: 0,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Sox,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Diameter,Sampling,Stack Temperature,%Oxygen,Gas Velocity,Flow rate,SOx Emission Actual O2,SOx Emission rate,Operation Hour,...,Flow rate,SOx Emission Actual O2,SOx Emission rate,Operation Hour,SOx Emission,NaN,NaN,NaN,NaN,NaN
3,Spray Dryer 1,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,15.72,1.3362,452,...,1416.666667,94.2,8.007,660,5284.62,NaN,NaN,NaN,NaN,NaN
4,Spray Dryer 2,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,18.34,1.5589,400,...,1416.666667,94.7,8.0495,436,3509.582,NaN,NaN,NaN,NaN,NaN
5,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2/12/2019,153.2,NaN,NaN,173.3333333,83.84,0.871936,384,...,154,92.4,0.853776,744,635.209344,NaN,NaN,NaN,NaN,NaN
6,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2/12/2019,159.1,NaN,NaN,188.3333333,84.71,0.957223,416,...,158.3333333,96.1,0.91295,744,679.2348,NaN,NaN,NaN,NaN,NaN
7,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2/12/2019,146.5,NaN,NaN,175.5,80.19,0.8444007,424,...,194.1666667,93.8,1.09277,744,813.02088,NaN,NaN,NaN,NaN,NaN
8,Glaze Kiln Exhaust Stack 1,0.8,29/9/2019,157.3,NaN,NaN,168.6666667,6.38,0.0645656,384,...,178.3333333,96.2,1.02934,744,765.82896,NaN,NaN,NaN,NaN,NaN
9,Glaze Kiln Exhaust Stack 2,0.8,29/9/2019,124.3,NaN,NaN,141.5,6.42,0.0545058,384,...,155.6666667,97.8,0.913452,744,679.608288,NaN,NaN,NaN,NaN,NaN
10,Glaze Kiln Exhaust Stack 3,0.8,29/9/2019,117.7,NaN,NaN,138,8.8,0.072864,416,...,202.3333333,92.9,1.127806,744,839.087664,NaN,NaN,NaN,NaN,NaN


# BƯỚC 3 - Chuyển đổi bảng dữ liệu từ ngang sang dọc, thêm cột 'period' và 'month'

In [ ]:
def reshape_emissions_data(df):
    """
    Hàm chuyển đổi bảng dữ liệu từ ngang sang dọc.
    Thêm cột 'period' (tên tháng) và cột 'month' (số thứ tự tháng).
    """
    # Ánh xạ tên tháng sang số
    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }

    stack_col_name = df.iloc[1, 0]
    diameter_col_name = df.iloc[1, 1]
    month_attributes = df.iloc[1, 2:11].tolist()
    data_df = df.iloc[2:].copy().reset_index(drop=True)

    melted_dfs = []
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    start_idx = 2

    for month_name in months:
        if start_idx >= df.shape[1]:
            break

        end_idx = start_idx + 9
        cols_to_extract = [0, 1] + list(range(start_idx, end_idx))
        temp_df = data_df.iloc[:, cols_to_extract].copy()

        # Đổi tên cột
        temp_df.columns = [stack_col_name, diameter_col_name] + month_attributes

        # THÊM CỘT DỮ LIỆU
        # Chèn 'period' (tên tháng) vào vị trí 0
        temp_df.insert(0, 'period', month_name)
        # Chèn 'month' (số tháng) vào vị trí 1 dựa trên bảng ánh xạ
        temp_df.insert(1, 'month', month_map[month_name])

        melted_dfs.append(temp_df)
        start_idx = end_idx

    # Nối và lọc dòng trống
    final_df = pd.concat(melted_dfs, ignore_index=True)
    final_df.dropna(subset=month_attributes, how='all', inplace=True)

    return final_df

# --- CHẠY TEST BƯỚC 3 ---
df_step3 = reshape_emissions_data(df_step2)
print(f"--- KẾT QUẢ BƯỚC 3: Kích thước: {df_step3.shape} ---")
display(df_step3.head(10))

--- KẾT QUẢ BƯỚC 3: Kích thước: (132, 13) ---


,period,month,Stack No.,Diameter,Sampling,Stack Temperature,%Oxygen,Gas Velocity,Flow rate,SOx Emission Actual O2,SOx Emission rate,Operation Hour,SOx Emission
0,Jan,1,Spray Dryer 1,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,15.72,1.3362,452,603.9624
1,Jan,1,Spray Dryer 2,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,18.34,1.5589,400,623.56
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2/12/2019,153.2,NaN,NaN,173.3333333,83.84,0.871936,384,334.823424
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2/12/2019,159.1,NaN,NaN,188.3333333,84.71,0.957223,416,398.204768
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2/12/2019,146.5,NaN,NaN,175.5,80.19,0.8444007,424,358.0258968
5,Jan,1,Glaze Kiln Exhaust Stack 1,0.8,29/9/2019,157.3,NaN,NaN,168.6666667,6.38,0.0645656,384,24.7931904
6,Jan,1,Glaze Kiln Exhaust Stack 2,0.8,29/9/2019,124.3,NaN,NaN,141.5,6.42,0.0545058,384,20.9302272
7,Jan,1,Glaze Kiln Exhaust Stack 3,0.8,29/9/2019,117.7,NaN,NaN,138,8.8,0.072864,416,30.311424
8,Jan,1,Glaze Kiln Exhaust Stack 4,0.8,2/12/2019,150.4,NaN,NaN,177.5,91.11,0.9703215,416,403.653744
9,Jan,1,Glaze Kiln Exhaust Stack 5,0.8,2/12/2019,120.1,NaN,NaN,178.3333333,92.18,0.986326,424,418.202224


# BƯỚC 4 - Đổi tên cột thành dạng snake_case và sắp xếp cột

In [ ]:
def clean_and_reorder_columns(df):
    """
    Hàm tự động chuẩn hóa tên cột sang định dạng _ (snake_case)
    và chuyển cột 'operation_hour' sang bên trái cột 'flow_rate'.
    """
    # Tạo một bản sao
    df_cleaned = df.copy()

    # Tự động đổi tên cột sang định dạng snake_case
    df_cleaned.columns = (
        df_cleaned.columns
        .str.strip()                         # Xóa khoảng trắng 2 đầu
        .str.lower()                         # Viết thường toàn bộ
        .str.replace('%oxygen', 'oxygen_pct', regex=False)   # Xóa ký tự %
        .str.replace('.', '', regex=False)   # Xóa dấu chấm
        .str.replace(' ', '_', regex=False)  # Thay khoảng trắng bằng dấu gạch dưới
    )

    # Đổi vị trí cột
    cols = df_cleaned.columns.tolist()

    # Kiểm tra xem có tồn tại 2 cột này không để tránh lỗi
    if 'operation_hour' in cols and 'flow_rate' in cols:
        cols.remove('operation_hour')
        flow_rate_index = cols.index('flow_rate')

        # Chèn 'operation_hour' vào ngay trước 'flow_rate'
        cols.insert(flow_rate_index, 'operation_hour')
        df_cleaned = df_cleaned[cols]

    return df_cleaned

# --- CHẠY TEST BƯỚC 4 ---
df_step4 = clean_and_reorder_columns(df_step3)
print(f"--- KẾT QUẢ BƯỚC 4: Kích thước: {df_step4.shape} ---")
display(df_step4.head(10))

--- KẾT QUẢ BƯỚC 4: Kích thước: (132, 13) ---


,period,month,stack_no,diameter,sampling,stack_temperature,oxygen_pct,gas_velocity,operation_hour,flow_rate,sox_emission_actual_o2,sox_emission_rate,sox_emission
0,Jan,1,Spray Dryer 1,1.3,2/12/2019,84.3,NaN,NaN,452,1416.666667,15.72,1.3362,603.9624
1,Jan,1,Spray Dryer 2,1.3,2/12/2019,87.1,NaN,NaN,400,1416.666667,18.34,1.5589,623.56
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2/12/2019,153.2,NaN,NaN,384,173.3333333,83.84,0.871936,334.823424
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2/12/2019,159.1,NaN,NaN,416,188.3333333,84.71,0.957223,398.204768
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2/12/2019,146.5,NaN,NaN,424,175.5,80.19,0.8444007,358.0258968
5,Jan,1,Glaze Kiln Exhaust Stack 1,0.8,29/9/2019,157.3,NaN,NaN,384,168.6666667,6.38,0.0645656,24.7931904
6,Jan,1,Glaze Kiln Exhaust Stack 2,0.8,29/9/2019,124.3,NaN,NaN,384,141.5,6.42,0.0545058,20.9302272
7,Jan,1,Glaze Kiln Exhaust Stack 3,0.8,29/9/2019,117.7,NaN,NaN,416,138,8.8,0.072864,30.311424
8,Jan,1,Glaze Kiln Exhaust Stack 4,0.8,2/12/2019,150.4,NaN,NaN,416,177.5,91.11,0.9703215,403.653744
9,Jan,1,Glaze Kiln Exhaust Stack 5,0.8,2/12/2019,120.1,NaN,NaN,424,178.3333333,92.18,0.986326,418.202224


In [ ]:
df_step4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   period                  132 non-null    object
 1   month                   132 non-null    int64 
 2   stack_no                132 non-null    object
 3   diameter                132 non-null    object
 4   sampling                132 non-null    object
 5   stack_temperature       132 non-null    object
 6   oxygen_pct              0 non-null      object
 7   gas_velocity            0 non-null      object
 8   operation_hour          132 non-null    object
 9   flow_rate               132 non-null    object
 10  sox_emission_actual_o2  132 non-null    object
 11  sox_emission_rate       132 non-null    object
 12  sox_emission            132 non-null    object
dtypes: int64(1), object(12)
memory usage: 13.5+ KB


##BƯỚC 5 - Xử lý số liệu và Ép kiểu

In [ ]:
def smart_auto_cast_and_fill(df, date_columns=['sampling']):
    """
    Hàm tự động toàn diện:
    1. Ép kiểu các cột ngày tháng được chỉ định.
    2. Nhận diện cột số để ép kiểu và điền 0.
    3. Giữ nguyên các cột định danh (chữ) và điền "N/A".
    """
    df_auto = df.copy()

    # Xử lý ép kiểu ngày tháng
    for col in date_columns:
        if col in df_auto.columns:
            df_auto[col] = pd.to_datetime(df_auto[col], format='%d/%m/%Y', errors='coerce')

    # Xử lý tự động ép kiểu số và điền dữ liệu khuyết cho các cột còn lại
    for col in df_auto.columns:
        if pd.api.types.is_datetime64_any_dtype(df_auto[col]):
            continue
        # Nếu cột đang trống hoàn toàn (toàn NaN) -> Mặc định đưa về số 0
        if df_auto[col].isna().all():
            df_auto[col] = 0.0
            continue
        converted = pd.to_numeric(df_auto[col], errors='coerce')
        valid_after = converted.notna().sum()

        if valid_after > 0:
            df_auto[col] = converted
            df_auto[col] = df_auto[col].fillna(0)
        else:

            df_auto[col] = df_auto[col].fillna("N/A")

    return df_auto

# --- CHẠY TEST BƯỚC 5 ---
df_step5 = smart_auto_cast_and_fill(df_step4)
print(f"--- KẾT QUẢ BƯỚC 5: Kích thước: {df_step5.shape} ---")
df_step5.info()
display(df_step5.head(10))

--- KẾT QUẢ BƯỚC 5: Kích thước: (132, 13) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   period                  132 non-null    object        
 1   month                   132 non-null    int64         
 2   stack_no                132 non-null    object        
 3   diameter                132 non-null    float64       
 4   sampling                132 non-null    datetime64[ns]
 5   stack_temperature       132 non-null    float64       
 6   oxygen_pct              132 non-null    float64       
 7   gas_velocity            132 non-null    float64       
 8   operation_hour          132 non-null    float64       
 9   flow_rate               132 non-null    float64       
 10  sox_emission_actual_o2  132 non-null    float64       
 11  sox_emission_rate       132 non-null    float64       
 12  sox_

,period,month,stack_no,diameter,sampling,stack_temperature,oxygen_pct,gas_velocity,operation_hour,flow_rate,sox_emission_actual_o2,sox_emission_rate,sox_emission
0,Jan,1,Spray Dryer 1,1.3,2019-12-02,84.3,0.0,0.0,452.0,1416.666667,15.72,1.336200,603.962400
1,Jan,1,Spray Dryer 2,1.3,2019-12-02,87.1,0.0,0.0,400.0,1416.666667,18.34,1.558900,623.560000
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2019-12-02,153.2,0.0,0.0,384.0,173.333333,83.84,0.871936,334.823424
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2019-12-02,159.1,0.0,0.0,416.0,188.333333,84.71,0.957223,398.204768
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2019-12-02,146.5,0.0,0.0,424.0,175.500000,80.19,0.844401,358.025897
5,Jan,1,Glaze Kiln Exhaust Stack 1,0.8,2019-09-29,157.3,0.0,0.0,384.0,168.666667,6.38,0.064566,24.793190
6,Jan,1,Glaze Kiln Exhaust Stack 2,0.8,2019-09-29,124.3,0.0,0.0,384.0,141.500000,6.42,0.054506,20.930227
7,Jan,1,Glaze Kiln Exhaust Stack 3,0.8,2019-09-29,117.7,0.0,0.0,416.0,138.000000,8.80,0.072864,30.311424
8,Jan,1,Glaze Kiln Exhaust Stack 4,0.8,2019-12-02,150.4,0.0,0.0,416.0,177.500000,91.11,0.970321,403.653744
9,Jan,1,Glaze Kiln Exhaust Stack 5,0.8,2019-12-02,120.1,0.0,0.0,424.0,178.333333,92.18,0.986326,418.202224


##Đóng gói Pipeline (Gom các hàm lại)

In [ ]:
def clean_table(df_raw):
    """
    Hàm tổng hợp xử lý df qua 5 bước làm sạch chuyên sâu.
    """
    df = df_raw.copy()
    df = clean_vietnamese_in_table(df)
    df = clean_data_custom(df)
    df = reshape_emissions_data(df)
    df = clean_and_reorder_columns(df)
    df = smart_auto_cast_and_fill(df, date_columns=['sampling'])

    return df

In [ ]:
# Danh sách các tên file cần xử lý
file_names = ['Emission_Dust.csv', 'Emission_Nox.csv', 'Emission_Sox.csv']
cleaned_dataframes = {}

for file in file_names:
    try:
        # Đọc file
        df_raw = pd.read_csv(file)
        # Gọi hàm clean_table
        df_cleaned = clean_table(df_raw)
        new_name = file.replace('.csv', '_Clean')
        cleaned_dataframes[new_name] = df_cleaned

        print(f"Đã xử lý xong: {new_name}")

    except Exception as e:
        print(f"Lỗi khi xử lý file {file}: {e}")

# Kiểm tra thử
display(cleaned_dataframes['Emission_Dust_Clean'])
display(cleaned_dataframes['Emission_Nox_Clean'])
display(cleaned_dataframes['Emission_Sox_Clean'])

Đã xử lý xong: Emission_Dust_Clean
Đã xử lý xong: Emission_Nox_Clean
Đã xử lý xong: Emission_Sox_Clean


,period,month,stack_no,diameter,sampling,stack_temperature,oxygen_pct,gas_velocity,operation_hour,flow_rate,dust_emission_actual_o2,dust_emission_rate,dust_emission
0,Jan,1,Spray Dryer 1,1.3,2019-12-02,84.3,0.0,0.0,452.0,1416.666667,41.0,3.485000,1575.220000
1,Jan,1,Spray Dryer 2,1.3,2019-12-02,87.1,0.0,0.0,400.0,1416.666667,43.0,3.655000,1462.000000
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2019-12-02,153.2,0.0,0.0,384.0,173.333333,52.0,0.540800,207.667200
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2019-12-02,159.1,0.0,0.0,416.0,188.333333,53.0,0.598900,249.142400
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2019-12-02,146.5,0.0,0.0,424.0,175.500000,50.0,0.526500,223.236000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Dec,12,Glaze Kiln Exhaust Stack 2,0.8,2020-08-20,124.7,0.0,0.0,744.0,155.666667,44.2,0.412828,307.144032
128,Dec,12,Glaze Kiln Exhaust Stack 3,0.8,2020-08-20,107.8,0.0,0.0,744.0,202.333333,47.9,0.581506,432.640464
129,Dec,12,Glaze Kiln Exhaust Stack 4,0.8,2020-08-20,111.4,0.0,0.0,624.0,174.166667,46.2,0.482790,301.260960
130,Dec,12,Glaze Kiln Exhaust Stack 5,0.8,2019-12-02,120.1,0.0,0.0,0.0,178.333333,52.5,0.561750,0.000000


,period,month,stack_no,diameter,sampling,stack_temperature,oxygen_pct,gas_velocity,operation_hour,flow_rate,sox_emission_actual_o2,sox_emission_rate,sox_emission
0,Jan,1,Spray Dryer 1,1.3,2019-12-02,84.3,0.0,0.0,452.0,1416.666667,15.72,1.336200,603.962400
1,Jan,1,Spray Dryer 2,1.3,2019-12-02,87.1,0.0,0.0,400.0,1416.666667,18.34,1.558900,623.560000
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2019-12-02,153.2,0.0,0.0,384.0,173.333333,83.84,0.871936,334.823424
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2019-12-02,159.1,0.0,0.0,416.0,188.333333,84.71,0.957223,398.204768
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2019-12-02,146.5,0.0,0.0,424.0,175.500000,80.19,0.844401,358.025897
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Dec,12,Glaze Kiln Exhaust Stack 2,0.8,2020-08-20,124.7,0.0,0.0,744.0,155.666667,97.80,0.913452,679.608288
128,Dec,12,Glaze Kiln Exhaust Stack 3,0.8,2020-08-20,107.8,0.0,0.0,744.0,202.333333,92.90,1.127806,839.087664
129,Dec,12,Glaze Kiln Exhaust Stack 4,0.8,2020-08-20,111.4,0.0,0.0,624.0,174.166667,97.10,1.014695,633.169680
130,Dec,12,Glaze Kiln Exhaust Stack 5,0.8,2019-12-02,120.1,0.0,0.0,0.0,178.333333,92.18,0.986326,0.000000


,period,month,stack_no,diameter,sampling,stack_temperature,oxygen_pct,gas_velocity,operation_hour,flow_rate,nox_emission_actual_o2,nox_emission_rate,nox_emission
0,Jan,1,Spray Dryer 1,1.3,2019-12-02,84.3,0.0,0.0,452.0,1416.666667,4.4,0.374000,169.048000
1,Jan,1,Spray Dryer 2,1.3,2019-12-02,87.1,0.0,0.0,400.0,1416.666667,4.6,0.391000,156.400000
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2019-12-02,153.2,0.0,0.0,384.0,173.333333,5.1,0.053040,20.367360
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2019-12-02,159.1,0.0,0.0,416.0,188.333333,4.6,0.051980,21.623680
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2019-12-02,146.5,0.0,0.0,424.0,175.500000,4.8,0.050544,21.430656
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Dec,12,Glaze Kiln Exhaust Stack 2,0.8,2020-08-20,124.7,0.0,0.0,744.0,155.666667,70.7,0.660338,491.291472
128,Dec,12,Glaze Kiln Exhaust Stack 3,0.8,2020-08-20,107.8,0.0,0.0,744.0,202.333333,75.7,0.918998,683.734512
129,Dec,12,Glaze Kiln Exhaust Stack 4,0.8,2020-08-20,111.4,0.0,0.0,624.0,174.166667,70.2,0.733590,457.760160
130,Dec,12,Glaze Kiln Exhaust Stack 5,0.8,2019-12-02,120.1,0.0,0.0,0.0,178.333333,4.3,0.046010,0.000000


# Xác định các cột dùng chung và gộp dữ liệu, sắp xếp các hàng theo tháng và tên

In [ ]:
def merge_emissions_data(list_dfs, labels=['_dust', '_nox', '_sox']):
    """
    Gộp dữ liệu, sắp xếp theo tháng và thứ tự ống khói (stack_no).
    """
    if not list_dfs:
        return None

    # Tự động xác định các cột chung
    common_cols = set(list_dfs[0].columns)
    for df in list_dfs[1:]:
        common_cols = common_cols.intersection(set(df.columns))

    common_cols = list(common_cols)
    print(f"--- Các cột chung dùng để gộp: ---\n{common_cols}\n")

    # Thực hiện gộp dữ liệu
    df_final = list_dfs[0]
    for i in range(1, len(list_dfs)):
        df_final = pd.merge(
            df_final,
            list_dfs[i],
            on=common_cols,
            how='outer',
            suffixes=(labels[i-1] if i==1 else "", labels[i])
        )

    # 3. Định nghĩa thứ tự ưu tiên cho cột stack_no
    stack_order = [
        'Spray Dryer 1', 'Spray Dryer 2',
        'Biscuit Kiln Exhaust Stack 1, 2', 'Biscuit Kiln Exhaust Stack 3, 4', 'Biscuit Kiln Exhaust Stack 5, 6',
        'Glaze Kiln Exhaust Stack 1', 'Glaze Kiln Exhaust Stack 2',
        'Glaze Kiln Exhaust Stack 3', 'Glaze Kiln Exhaust Stack 4',
        'Glaze Kiln Exhaust Stack 5', 'Glaze Kiln Exhaust Stack 6'
    ]

    if 'stack_no' in df_final.columns:
        df_final['stack_no'] = pd.Categorical(df_final['stack_no'], categories=stack_order, ordered=True)

    sort_columns = []
    if 'month' in df_final.columns: sort_columns.append('month')
    if 'stack_no' in df_final.columns: sort_columns.append('stack_no')

    if sort_columns:
        df_final = df_final.sort_values(by=sort_columns).reset_index(drop=True)
        print(f"--- Đã sắp xếp dữ liệu theo: {', '.join(sort_columns)} ---")

    # Thông báo cột riêng biệt
    unique_cols = [c for c in df_final.columns if c not in common_cols]
    print(f"--- Các cột giá trị riêng: {unique_cols} ---")

    return df_final

# --- THỰC THI ---
dataframes_to_merge = [cleaned_dataframes['Emission_Dust_Clean'], cleaned_dataframes['Emission_Nox_Clean'], cleaned_dataframes['Emission_Sox_Clean']]
df_final = merge_emissions_data(dataframes_to_merge)
df_final.info()
display(df_final)

--- Các cột chung dùng để gộp: ---
['diameter', 'oxygen_pct', 'period', 'stack_no', 'sampling', 'flow_rate', 'gas_velocity', 'month', 'operation_hour', 'stack_temperature']

--- Đã sắp xếp dữ liệu theo: month, stack_no ---
--- Các cột giá trị riêng: ['dust_emission_actual_o2', 'dust_emission_rate', 'dust_emission', 'sox_emission_actual_o2', 'sox_emission_rate', 'sox_emission', 'nox_emission_actual_o2', 'nox_emission_rate', 'nox_emission'] ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   period                   132 non-null    object        
 1   month                    132 non-null    int64         
 2   stack_no                 132 non-null    category      
 3   diameter                 132 non-null    float64       
 4   sampling                 132 non-null    datetime64[ns]
 5   stack_temperatur

,period,month,stack_no,diameter,sampling,stack_temperature,oxygen_pct,gas_velocity,operation_hour,flow_rate,dust_emission_actual_o2,dust_emission_rate,dust_emission,sox_emission_actual_o2,sox_emission_rate,sox_emission,nox_emission_actual_o2,nox_emission_rate,nox_emission
0,Jan,1,Spray Dryer 1,1.3,2019-12-02,84.3,0.0,0.0,452.0,1416.666667,41.0,3.485000,1575.220000,15.72,1.336200,603.962400,4.4,0.374000,169.048000
1,Jan,1,Spray Dryer 2,1.3,2019-12-02,87.1,0.0,0.0,400.0,1416.666667,43.0,3.655000,1462.000000,18.34,1.558900,623.560000,4.6,0.391000,156.400000
2,Jan,1,"Biscuit Kiln Exhaust Stack 1, 2",0.8,2019-12-02,153.2,0.0,0.0,384.0,173.333333,52.0,0.540800,207.667200,83.84,0.871936,334.823424,5.1,0.053040,20.367360
3,Jan,1,"Biscuit Kiln Exhaust Stack 3, 4",0.8,2019-12-02,159.1,0.0,0.0,416.0,188.333333,53.0,0.598900,249.142400,84.71,0.957223,398.204768,4.6,0.051980,21.623680
4,Jan,1,"Biscuit Kiln Exhaust Stack 5, 6",0.8,2019-12-02,146.5,0.0,0.0,424.0,175.500000,50.0,0.526500,223.236000,80.19,0.844401,358.025897,4.8,0.050544,21.430656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Dec,12,Glaze Kiln Exhaust Stack 2,0.8,2020-08-20,124.7,0.0,0.0,744.0,155.666667,44.2,0.412828,307.144032,97.80,0.913452,679.608288,70.7,0.660338,491.291472
128,Dec,12,Glaze Kiln Exhaust Stack 3,0.8,2020-08-20,107.8,0.0,0.0,744.0,202.333333,47.9,0.581506,432.640464,92.90,1.127806,839.087664,75.7,0.918998,683.734512
129,Dec,12,Glaze Kiln Exhaust Stack 4,0.8,2020-08-20,111.4,0.0,0.0,624.0,174.166667,46.2,0.482790,301.260960,97.10,1.014695,633.169680,70.2,0.733590,457.760160
130,Dec,12,Glaze Kiln Exhaust Stack 5,0.8,2019-12-02,120.1,0.0,0.0,0.0,178.333333,52.5,0.561750,0.000000,92.18,0.986326,0.000000,4.3,0.046010,0.000000


In [ ]:
# Save the final DataFrame to a CSV file
df_final.to_csv('Emission _Cleaned.csv', index=False)